# Cell Annotation with Napari

Input
- one image (split tif, PCNA channel)
- one mask (png output from cellpose)

Output
- new mask with annotated labels
- labels range from 1-5 for cell cycle phase (S-early/mid/late, G1G2, ambiguous)

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from skimage.measure import label, regionprops
from tifffile import imread as tifread
from tifffile import imwrite
from skimage.io import imread as pngread
import napari
import os

In [66]:
img_num = '003'
img_pos = '4'

image_path = '/Users/josefin/PhD/weihua_UHRF1_CellCycle/images/2603011/tif_IllCorr/wt_UHRF1488_PCNA561_DAPI_' + img_num + '_pos' + img_pos + '_ch1.tif'
mask_path = '/Users/josefin/PhD/weihua_UHRF1_CellCycle/results/segmentation_nuclei_IllCorr/wt_UHRF1488_PCNA561_DAPI_' + img_num + '_pos' + img_pos + '_ch2_cp_masks.png'
out_path = '/Users/josefin/PhD/weihua_UHRF1_CellCycle/cell_annotations'

if not Path(out_path).exists():
    Path(out_path).mkdir()

In [ ]:
# get both image and mask
image = tifread(image_path)
mask = pngread(mask_path)

# regionprops of mask for bbox and labels
props = regionprops(mask)

shapes = []
labels=[]

for i, p in enumerate(props, start=1):
    minc, minr, maxc, maxr = p.bbox # r=c
    shapes.append(np.array([[minc, minr], [maxc, minr], [maxc, maxr], [minc, maxr]]))
    labels.append(p.label)

    
viewer = napari.Viewer()
viewer.add_image(image, name="image", colormap="gray", gamma=0.5) # immediately decrease gamma 
viewer.add_labels(mask, name="mask", visible=False) # keep mask for double checking
viewer.add_shapes( 
    shapes,
    shape_type="polygon",
    edge_color="yellow",
    face_color="transparent",
    name="bbox",
    opacity=0.8,
)

# one layer per phase/label
early_points = viewer.add_points(None, ndim=2, size=10, face_color="#fd8a2b", name='early')
mid_points = viewer.add_points(None, ndim=2, size=10, face_color="#9bd829", name='mid')
late_points = viewer.add_points(None, ndim=2, size=10, face_color="#39f3e4", name='late')
g1g2_points = viewer.add_points(None, ndim=2, size=10, face_color="#5b5dff", name='g1g2')
ambig_points = viewer.add_points(None, ndim=2, size=10, face_color="#ec2ce3", name='ambiguous')


napari.run()


In [ ]:
new_mask = np.zeros_like(mask, dtype=np.int32)
label_mapping = {}

# define class IDs (not matching Yazi's: everything +1, because 0=background)
class_early = 1
class_mid = 4
class_late = 3
class_g1g2 = 2
class_ambig = 5

# early
for point in early_points.data:
    r, c = map(int, point)
    label = mask[r, c]
    label_mapping[label] = class_early

# mid
for point in mid_points.data:
    r, c = map(int, point)
    label = mask[r, c]
    label_mapping[label] = class_mid


# late
for point in late_points.data:
    r, c = map(int, point)
    label = mask[r, c]
    label_mapping[label] = class_late

# G1G2
for point in g1g2_points.data:
    r, c = map(int, point)
    label = mask[r, c]
    label_mapping[label] = class_g1g2

# ambiguous
for point in ambig_points.data:
    r, c = map(int, point)
    label = mask[r, c]
    label_mapping[label] = class_ambig

# changes labels of old mask to the newly annotated 
for original_label, new_label in label_mapping.items():
    new_mask[mask == original_label] = new_label




In [ ]:
# make df from labels, label-mapping and shapes (=bboxes)
rows = []

for label, bbox in zip(labels, shapes):
    rows.append({
        "cellpose_label": label,
        "cellcycle_label": label_mapping.get(label, 0),
        "y_min": bbox[:, 0].min(),
        "x_min": bbox[:, 1].min(),
        "y_max": bbox[:, 0].max(),
        "x_max": bbox[:, 1].max(),
    })

df = pd.DataFrame(rows)

base_name = Path(image_path).stem

class_names = {
    0: 'not_mapped',
    1: 'EarlyS',
    2: 'G1G2',
    3: 'LateS',
    4: 'MidS', 
    5: 'ambiguous'}

for class_id, class_name in class_names.items():
    os.makedirs(os.path.join(out_path, class_name), exist_ok=True)

for idx, row in df.iterrows():
    y0, x0 = int(row["y_min"])-10, int(row["x_min"])-10
    y1, x1 = int(row["y_max"])+10, int(row["x_max"])+10

    # TODO: more elegant solution for "edge-cells"
    if y0 < 0:
        y0 = int(row["y_min"])
    if x0 < 0:
        x0 = int(row["x_min"])
    if y1 > 1024:
        y1 = int(row["y_max"])
    if x1 > 1024:
        x1 = int(row["x_max"])
        
    patch = image[y0:y1, x0:x1]

    # Normalize to 8-bit RGB
    patch = patch.astype(np.float32)
    patch = (patch - patch.min()) / (patch.max() - patch.min() + 1e-5) * 255.0
    # 8 bit
    patch_uint8 = patch.astype(np.uint8)
    # rgb
    patch_rgb = np.stack([patch_uint8] * 3, axis=-1)

    filename = f"{base_name}_cell{row['cellpose_label']}.tif"

    class_id = row["cellcycle_label"]
    class_name = class_names.get(class_id, "unknown")

    save_path_crops = os.path.join(out_path, class_name, filename)
    save_path_csv = os.path.join(out_path, base_name + '_results.csv')


    imwrite(save_path_crops, patch_rgb)
    df.to_csv(save_path_csv)
